# QuantLab Interactive Dashboard

This notebook provides an interactive dashboard for visualizing QuantLab's market analysis, signals, and portfolio allocation.

Run each cell in order to see the current market state and recommended allocation.

In [ ]:
# Cell 1: Imports and setup
import sys
sys.path.insert(0, '..')

from quantlab.config import QuantLabConfig, ASSET_FAMILIES
from quantlab.data.ingestion import download_latest_data
from quantlab.data.storage import DataStore
from quantlab.data.validation import validate_ohlcv
from quantlab.features.engine import FeatureEngine
from quantlab.signals.engine import SignalEngine
from quantlab.risk.manager import RiskManager
from quantlab.portfolio.allocator import PortfolioAllocator
from quantlab.strategies.regime import HMMRegimeStrategy
from quantlab.strategies.dynamic import DynamicCovarianceStrategy
from quantlab.strategies.composer import ComposerRSIStrategy
from quantlab.strategies.mean_reversion import MeanReversionStrategy
from quantlab.strategies.combined import CombinedStrategy
from quantlab.utils.metrics import calculate_all_metrics
from quantlab.utils.monte_carlo import MonteCarloSimulator

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

config = QuantLabConfig()
plt.style.use('seaborn-v0_8-darkgrid')
print('QuantLab v1.0.0 - Dashboard initialized')
print(f'Asset families: {list(ASSET_FAMILIES.keys())}')

## Cell 2: Download Data
Download the latest OHLCV data for all configured asset families.

In [ ]:
# Cell 2: Download data
print('Downloading latest market data...')
raw_data = download_latest_data(config)
print(f'Downloaded {len(raw_data)} rows of OHLCV data')
print(f'Date range: {raw_data.index.min()} to {raw_data.index.max()}')
print(f'Tickers: {raw_data.columns.get_level_values(0).unique().tolist() if isinstance(raw_data.columns, pd.MultiIndex) else raw_data.columns.tolist()}')
raw_data.tail()

## Cell 3: Compute Features
Compute all technical features: returns, volatility, RSI, drawdown, trend z-score, momentum, and volatility regime.

In [ ]:
# Cell 3: Compute features
if isinstance(raw_data.columns, pd.MultiIndex):
    close_prices = raw_data.xs('Close', level=1, axis=1)
else:
    close_prices = raw_data.copy()

feature_engine = FeatureEngine(prices=close_prices)
features = feature_engine.compute_all()
print(f'Computed features shape: {features.shape}')
print(f'Feature columns: {list(features.columns[:10])}...')
features.tail()

## Cell 4: Run Strategies and Collect Signals
Register all strategies and run them to generate signals for each asset family.

In [ ]:
# Cell 4: Run strategies and collect signals
signal_engine = SignalEngine(config=config)

# Register family-specific strategies
for family_name, assets in ASSET_FAMILIES.items():
    signal_engine.register_strategy(HMMRegimeStrategy(family_name=family_name, assets=assets))
    signal_engine.register_strategy(DynamicCovarianceStrategy(family_name=family_name, assets=assets))
    signal_engine.register_strategy(CombinedStrategy(family_name=family_name, assets=assets))

# Register global strategies
signal_engine.register_strategy(ComposerRSIStrategy())
signal_engine.register_strategy(MeanReversionStrategy())

print(f'Running {len(signal_engine._strategies)} strategies...')
signals = signal_engine.run_all(features=features, prices=close_prices)
print(f'Generated {len(signals)} signals')
for sig in signals:
    print(f"  {sig['strategy']:25s} | {sig['family']:15s} | {sig['risk_state']:7s} | conf={sig['confidence']:.2f}")

## Cell 5: Risk Check and Allocate
Apply risk management checks and compute final portfolio allocation.

In [ ]:
# Cell 5: Risk check and allocate
risk_manager = RiskManager(config=config)
allocator = PortfolioAllocator(config=config)

# Apply risk checks to each signal
approved_signals = []
for signal in signals:
    result = risk_manager.check(signal)
    if result['approved']:
        approved_signals.append(result.get('modified_signal', signal))
        print(f"APPROVED: {signal['strategy']}/{signal['family']}")
    else:
        print(f"REJECTED: {signal['strategy']}/{signal['family']} - {result.get('reasons', [])}")

print(f'\nApproved {len(approved_signals)} of {len(signals)} signals')

# Allocate portfolio
allocation = allocator.allocate(
    signals=approved_signals if approved_signals else signals,
    features=features
)

print(f"\nFinal Allocation (risk={allocation['risk_score']}, conf={allocation['confidence']:.2f}):")
for ticker, weight in sorted(allocation['final_weights'].items(), key=lambda x: x[1], reverse=True):
    if weight > 0.001:
        print(f'  {ticker:8s}: {weight * 100:6.2f}%')

print(f"\nReason: {allocation['reason']}")

## Cell 6: Display Current Market State
Show latest prices and trend indicators.

In [ ]:
# Cell 6: Display current market state
latest = close_prices.iloc[-1]
prev = close_prices.iloc[-2] if len(close_prices) > 1 else close_prices.iloc[-1]

print('=' * 60)
print('CURRENT MARKET STATE')
print('=' * 60)
print(f'Date: {close_prices.index[-1].strftime("%Y-%m-%d")}')
print()

for ticker in sorted(close_prices.columns):
    price = latest[ticker]
    change = (price - prev[ticker]) / prev[ticker] * 100 if prev[ticker] != 0 else 0
    change_symbol = '+' if change >= 0 else ''
    print(f'{ticker:8s}: ${price:>10.2f}  ({change_symbol}{change:.2f}%)')

# VIX indicator (if available)
if 'VIX' in close_prices.columns:
    vix = latest['VIX']
    vix_level = 'Low (complacent)' if vix < 15 else 'Normal' if vix < 25 else 'High (fearful)' if vix < 35 else 'Extreme (panic)'
    print(f'\nVIX Level: {vix:.2f} - {vix_level}')

# Trend summary using feature data
trend_cols = [c for c in features.columns if 'trend_z20' in str(c)]
if trend_cols:
    avg_trend = np.mean([features.iloc[-1][c] for c in trend_cols if pd.notna(features.iloc[-1][c])])
    direction = 'UP' if avg_trend > 0.5 else 'DOWN' if avg_trend < -0.5 else 'NEUTRAL'
    print(f'Overall Trend: {direction} (z-score: {avg_trend:.2f})')

print('=' * 60)

## Cell 7: Display Signals Table
Comprehensive table of all strategy signals.

In [ ]:
# Cell 7: Display signals table
signals_df = pd.DataFrame([
    {
        'Date': s['date'],
        'Strategy': s['strategy'],
        'Family': s['family'],
        'Confidence': s['confidence'],
        'Risk State': s['risk_state'],
        'Primary Holding': max(s['weights'].items(), key=lambda x: x[1])[0] if s['weights'] else 'N/A',
        'Primary Weight': max(s['weights'].values()) if s['weights'] else 0,
        'Reason': s['reason'][:60] + '...' if len(s['reason']) > 60 else s['reason']
    }
    for s in signals
])

# Color coding by risk state
def color_risk(val):
    colors = {'low': 'green', 'medium': 'orange', 'high': 'red', 'extreme': 'purple'}
    return f'color: {colors.get(val, "black")}'

styled = signals_df.style.applymap(color_risk, subset=['Risk State'])
styled

## Cell 8: Plot Suggested Allocation Pie Chart

In [ ]:
# Cell 8: Plot suggested allocation pie chart
weights = allocation['final_weights']
filtered = {k: v for k, v in weights.items() if v > 0.001}

if filtered:
    sorted_items = sorted(filtered.items(), key=lambda x: x[1], reverse=True)
    labels = [ticker for ticker, _ in sorted_items]
    sizes = [weight for _, weight in sorted_items]

    fig, ax = plt.subplots(figsize=(8, 8))
    colors = plt.cm.tab20(np.linspace(0, 1, len(labels)))

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        autopct='%1.1f%%',
        startangle=90,
        colors=colors,
        textprops={'fontsize': 10},
    )
    ax.set_title('Suggested Portfolio Allocation', fontsize=14, fontweight='bold')
    plt.setp(autotexts, size=9, weight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No allocation to display (all weights are zero).')

## Cell 9: Plot Regime Probabilities Bar Chart

In [ ]:
# Cell 9: Plot regime probabilities bar chart
regime_probs = {}
for signal in signals:
    if 'hmm' in signal.get('strategy', '').lower():
        metadata = signal.get('metadata', {})
        if 'regime_probs' in metadata:
            regime_probs = metadata['regime_probs']
            break
        elif 'regime' in signal:
            regime_probs = {signal['regime']: 1.0}
            break

if not regime_probs:
    regime_probs = {'Bull': 0.33, 'Neutral': 0.34, 'Bear': 0.33}

# Normalize
total = sum(regime_probs.values())
regime_probs = {k: v / total for k, v in regime_probs.items()}

fig, ax = plt.subplots(figsize=(8, 5))
regimes = list(regime_probs.keys())
probs = [regime_probs[r] * 100 for r in regimes]
colors = ['#16a34a', '#d97706', '#dc2626'][:len(regimes)]

bars = ax.bar(regimes, probs, color=colors, edgecolor='white', linewidth=2)
ax.set_ylabel('Probability (%)', fontsize=11)
ax.set_title('HMM Regime Probabilities', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')

for bar, prob in zip(bars, probs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f'{prob:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Cell 10: Display Risk Score
Visual indicator of the overall risk assessment.

In [ ]:
# Cell 10: Display risk score
from IPython.display import HTML, display

risk_score = allocation.get('risk_score', 'unknown')
confidence = allocation.get('confidence', 0.0)

risk_colors = {
    'low': '#16a34a',
    'medium': '#d97706',
    'high': '#dc2626',
    'extreme': '#7c3aed'
}
risk_bg_colors = {
    'low': '#dcfce7',
    'medium': '#fef3c7',
    'high': '#fee2e2',
    'extreme': '#ede9fe'
}

color = risk_colors.get(risk_score, '#6b7280')
bg_color = risk_bg_colors.get(risk_score, '#f3f4f6')

html = f'''
<div style='background: {bg_color}; border-left: 6px solid {color}; padding: 20px; border-radius: 8px;'>
    <h3 style='margin: 0 0 10px 0; color: {color};'>Risk Assessment: {risk_score.upper()}</h3>
    <p style='margin: 0; font-size: 1.1em;'>
        Confidence: <strong>{confidence * 100:.1f}%</strong> | 
        Approved Signals: <strong>{len(approved_signals)}/{len(signals)}</strong>
    </p>
</div>
'''
display(HTML(html))

# Risk breakdown
risk_counts = pd.Series([s['risk_state'] for s in signals]).value_counts()
print('\nSignal Risk Distribution:')
for state, count in risk_counts.items():
    print(f'  {state:10s}: {count} signals')

## Cell 11: Plot Correlation Heatmap

In [ ]:
# Cell 11: Plot correlation heatmap
returns = close_prices.pct_change().dropna()

if len(returns) > 20:
    corr = returns.corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn',
        center=0,
        vmin=-1, vmax=1,
        square=True,
        linewidths=0.5,
        ax=ax
    )
    ax.set_title('Asset Return Correlations (60-day)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient data for correlation heatmap (need > 20 days).')

## Cell 12: Plot Equity Curve
Strategy equity curve compared to benchmark (if backtest data available).

In [ ]:
# Cell 12: Plot equity curve
if len(returns) > 1:
    # Compute cumulative returns for equal-weighted benchmark
    equal_weight_returns = returns.mean(axis=1)
    cumulative = (1 + equal_weight_returns).cumprod()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(cumulative.index, cumulative.values, label='Equal-Weight Portfolio', linewidth=2)

    # Add SPY if available
    if 'SPY' in returns.columns:
        spy_cumulative = (1 + returns['SPY']).cumprod()
        ax.plot(spy_cumulative.index, spy_cumulative.values, label='SPY Benchmark', linewidth=2, linestyle='--')

    ax.set_title('Equity Curve', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Cumulative Return', fontsize=11)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Print summary stats
    total_return = cumulative.iloc[-1] - 1
    annualized = (1 + total_return) ** (252 / len(returns)) - 1 if len(returns) > 1 else 0
    vol = equal_weight_returns.std() * np.sqrt(252)
    sharpe = annualized / vol if vol > 0 else 0
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    max_dd = drawdown.min()

    print(f'Equal-Weight Portfolio Stats:')
    print(f'  Total Return: {total_return * 100:.2f}%')
    print(f'  Annualized Return: {annualized * 100:.2f}%')
    print(f'  Volatility: {vol * 100:.2f}%')
    print(f'  Sharpe Ratio: {sharpe:.2f}')
    print(f'  Max Drawdown: {max_dd * 100:.2f}%')
else:
    print('Insufficient data for equity curve.')

## Cell 13: Plot Drawdown Chart

In [ ]:
# Cell 13: Plot drawdown chart
if len(returns) > 1:
    equal_weight_returns = returns.mean(axis=1)
    cumulative = (1 + equal_weight_returns).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max * 100

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
    ax.plot(drawdown.index, drawdown.values, color='red', linewidth=1)
    ax.set_title('Drawdown Chart', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Drawdown (%)', fontsize=11)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Additional metrics
    print(f'\nDrawdown Statistics:')
    print(f'  Max Drawdown: {drawdown.min():.2f}%')
    print(f'  Current Drawdown: {drawdown.iloc[-1]:.2f}%')
    print(f'  Days in Drawdown: {(drawdown < -0.1).sum()}')
else:
    print('Insufficient data for drawdown chart.')

---

## Dashboard Complete

Run the cells above to refresh the dashboard with the latest data.

**Note:** This is for informational purposes only. Not investment advice.